# Davinci Notebook 10 — Pretraining Optimization & Mini-Run

Practical training optimization options, gradient checkpointing, corpus mixture sampling, packed token blocks, BF16/FP16 selection, checkpointing, anomaly detection, and a tiny end-to-end validation run. This is a prototype scaffold: integrate the model class exported from Notebook 5/9 and supply prepared token data.

In [ ]:
import os, math, random, time, json, torch
from torch.utils.data import Dataset, DataLoader
print('torch',torch.__version__,'cuda',torch.cuda.is_available())


## 1. Hardware-aware precision and memory report

In [ ]:
def runtime_report():
 print('device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
 if torch.cuda.is_available(): print('VRAM total GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
runtime_report()
def amp_dtype(): return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
AMP_DTYPE=amp_dtype(); print('preferred AMP dtype:',AMP_DTYPE)


## 2. Packed token-block dataset
The loader consumes already-tokenized document lists. Packing across document boundaries is optional; use boundary markers if configured and mask cross-document prediction where desired. This baseline uses continuous concatenation and should be paired with a documented EOS token.

In [ ]:
class TokenBlockDataset(Dataset):
 def __init__(self,documents,seq_len):
  self.tokens=[]
  for doc in documents: self.tokens.extend(list(map(int,doc)))
  self.seq_len=seq_len; self.blocks=len(self.tokens)//seq_len
 def __len__(self): return self.blocks
 def __getitem__(self,i):
  start=i*self.seq_len; block=self.tokens[start:start+self.seq_len]
  return torch.tensor(block,dtype=torch.long)


## 3. Source-weighted sampling helper
Use weights to balance source datasets rather than letting the largest source dominate. Set weights based on measured quality, language coverage, licensing, and dedup overlap—not intuition alone.

In [ ]:
def sample_source_ids(source_sizes,weights,n_samples,seed=42):
 assert len(source_sizes)==len(weights) and all(w>=0 for w in weights) and sum(weights)>0
 rng=random.Random(seed); probs=[w/sum(weights) for w in weights]; out=[]
 for _ in range(n_samples):
  src=rng.choices(range(len(source_sizes)),weights=probs,k=1)[0]; idx=rng.randrange(source_sizes[src]); out.append((src,idx))
 return out


## 4. Training utilities: checkpointing, clipping, and save/load
PyTorch activation checkpointing trades compute for reduced activation memory. Use non-reentrant checkpointing where appropriate; checkpointing should be validated against the architecture and torch version.

In [ ]:
from torch.utils.checkpoint import checkpoint
def checkpoint_block(block,x,pos,enabled):
 if enabled and x.requires_grad: return checkpoint(lambda inp:block(inp,pos),x,use_reentrant=False)
 return block(x,pos)
def save_training_checkpoint(path,model,optimizer,step,config,scaler=None):
 os.makedirs(os.path.dirname(path) or '.',exist_ok=True); payload={'model':model.state_dict(),'optimizer':optimizer.state_dict(),'step':step,'config':config}
 if scaler is not None: payload['scaler']=scaler.state_dict()
 torch.save(payload,path)
def load_training_checkpoint(path,model,optimizer,scaler=None,map_location='cpu'):
 payload=torch.load(path,map_location=map_location,weights_only=False); model.load_state_dict(payload['model']); optimizer.load_state_dict(payload['optimizer'])
 if scaler is not None and 'scaler' in payload: scaler.load_state_dict(payload['scaler'])
 return payload['step'],payload.get('config',{})


## 5. Optimizer and scheduler factory

In [ ]:
def cosine_lr(step,total,warmup,peak,min_lr):
 if step<warmup: return peak*(step+1)/max(1,warmup)
 ratio=min(1,(step-warmup)/max(1,total-warmup)); return min_lr+(peak-min_lr)*.5*(1+math.cos(math.pi*ratio))


## 6. Generic mini-training loop
Supply a model returning logits `[B,T,V]`. Uses shifted causal loss, gradient accumulation, AMP, clipping, and periodic checkpointing. For architecture-level gradient checkpointing, wire `checkpoint_block` into the decoder loop.

In [ ]:
def mini_train(model,loader,steps,lr=3e-4,accum=1,clip=1.0,amp=True,save_path='mini_checkpoint.pt'):
 device=next(model.parameters()).device; opt=make_optimizer(model,lr); scaler=torch.amp.GradScaler('cuda',enabled=(amp and device.type=='cuda' and AMP_DTYPE==torch.float16)); model.train(); it=iter(loader); logs=[]
 for step in range(steps):
  opt.zero_grad(set_to_none=True); running=0.
  for _ in range(accum):
   try: batch=next(it)
   except StopIteration: it=iter(loader); batch=next(it)
   batch=batch.to(device); ctx=torch.autocast(device_type=device.type,dtype=AMP_DTYPE,enabled=(amp and device.type=='cuda'))
   with ctx: logits=model(batch); loss= torch.nn.functional.cross_entropy(logits[:,:-1,:].float().reshape(-1,logits.size(-1)),batch[:,1:].reshape(-1))/accum
   scaler.scale(loss).backward(); running+=loss.item()*accum
  scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),clip); scaler.step(opt); scaler.update(); logs.append(running/accum)
  if (step+1)%max(1,steps//5)==0: print(f'{step+1}/{steps} loss={logs[-1]:.4f}')
 save_training_checkpoint(save_path,model,opt,step+1,{'steps':steps,'lr':lr},scaler)
 return logs


## 7. Tiny smoke test (synthetic data)
This test uses a simple tiny language model, not the Davinci model, to validate the generic training utilities without requiring the full architecture or dataset.

In [ ]:
class TinyLM(torch.nn.Module):
 def __init__(self,vocab=64,d=32): super().__init__(); self.e=torch.nn.Embedding(vocab,d); self.l=torch.nn.Linear(d,vocab)
 def forward(self,x): return self.l(self.e(x))
toy=torch.randint(0,64,(128,32)); loader=DataLoader(toy,batch_size=8,shuffle=True)
tiny=TinyLM().to('cuda' if torch.cuda.is_available() else 'cpu'); logs=mini_train(tiny,loader,steps=3,amp=False,save_path='/tmp/davinci_mini_test.pt'); print('smoke-test losses:',logs)


## 8. Before full-scale pretraining
- Verify the tokenizer and data manifests, source licenses, and deduplication reports.
- Use an explicit train/validation split at document/source level to prevent leakage.
- Confirm token blocks include EOS and consider masking across concatenated document boundaries.
- Measure batch size and sequence length before scaling; monitor GPU memory.
- Consider gradient checkpointing, optimizer state memory, and efficient attention separately.
- Validate checkpoint resume with a real model and optimizer.
- Avoid claiming training quality from the synthetic smoke test.